# Module 2 — first LSTM (Wikipedia pageviews)

Same task as ARIMA in `forecasting/evaluate.py`: 15-min load, train on all but the last day,
predict the held-out last day (96 steps) in one shot, score with the same `errors()`.
Run in Colab (Runtime → Run all). CPU runtime is enough; takes a few minutes.

In [ ]:
import os, sys
REPO, BRANCH = "https://github.com/mddragon18/cloud-oracle", "main"
if "google.colab" in sys.modules:
    if not os.path.exists("cloud-oracle"):
        !git clone -q -b {BRANCH} {REPO} cloud-oracle
    os.chdir("cloud-oracle")
sys.path.insert(0, os.getcwd())

import numpy as np
import tensorflow as tf
from forecasting.loaders import wikipedia
from forecasting.forecast import _frame, _per_minute, errors

tf.keras.utils.set_random_seed(0)
STEP, PERIOD = 15, 96  # 15-min steps, 96 per day

In [ ]:
# 14 days of en.wikipedia hourly pageviews -> C1 -> 15-min requests_per_min
c1 = wikipedia(start="20261001", end="20261014")
y = _frame(_per_minute(c1.to_dict("records")), STEP)["rpm"].to_numpy()
train, test = y[:-PERIOD], y[-PERIOD:]
lo, hi = train.min(), train.max()  # scale with train stats only (no peeking at the held-out day)
scale = lambda v: (v - lo) / (hi - lo)
print(len(y), "steps;", len(train), "train,", len(test), "held out")

In [ ]:
# windows: previous day (96 steps) -> next day (96 steps)
s = scale(train)
X = np.stack([s[i:i + PERIOD] for i in range(len(s) - 2 * PERIOD + 1)])[..., None]
Y = np.stack([s[i + PERIOD:i + 2 * PERIOD] for i in range(len(s) - 2 * PERIOD + 1)])
print(X.shape, Y.shape)

model = tf.keras.Sequential([
    tf.keras.Input((PERIOD, 1)),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(PERIOD),
])
model.compile(optimizer="adam", loss="mse")
model.fit(X, Y, epochs=60, batch_size=32, validation_split=0.15, verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)])

In [ ]:
pred = model.predict(scale(train[-PERIOD:])[None, :, None], verbose=0)[0] * (hi - lo) + lo
lstm = errors(test, pred)
print("LSTM ", {k: round(v, 2) for k, v in lstm.items()})
print("ARIMA on the same day: see forecasting/results/accuracy.md (wikipedia row)")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3))
plt.plot(test, label="actual"); plt.plot(pred, label=f"LSTM (MAPE {lstm['mape_pct']:.1f}%)")
plt.legend(); plt.title("Wikipedia held-out day"); plt.ylabel("req/min"); plt.show()

In [ ]:
# Save. Phase III: upload to S3 for Lambda; for now Google Drive when in Colab.
path = "lstm_wikipedia.keras"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/cloud-oracle", exist_ok=True)
    path = "/content/drive/MyDrive/cloud-oracle/lstm_wikipedia.keras"
model.save(path)
print("saved", path)